# 07 — Gradient Boosting Models

Project: E-Waste Profitability Prediction
Group: 2026-AI-14
Module: IT3091 — Machine Learning

Regression: GradientBoostingRegressor
Classification: GradientBoostingClassifier

Gradient Boosting builds trees sequentially, with later trees
improving the current ensemble's predictions.

Validation:
- Preserve the existing train/test split.
- Use the same five-fold settings as previous notebooks.
- Fit preprocessing inside each CV training fold.
- Compare raw-only and engineered feature sets.
- Keep test evaluation pending until model selection is recorded.

Batch_Priority represents derived profit tiers.

Import libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import platform
import sklearn

from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import (
    GradientBoostingRegressor,
    GradientBoostingClassifier
)

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    GridSearchCV
)

pd.set_option("display.max_columns", None)

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
scikit-learn: 1.6.1


File Upload

In [ ]:
uploaded = files.upload()

Saving train_features.csv to train_features.csv
Saving test_features.csv to test_features.csv
Saving feature_config.json to feature_config.json


In [ ]:
required_files = [
    "train_features.csv",
    "test_features.csv",
    "feature_config.json"
]

for filename in required_files:
    assert filename in uploaded, f"Missing file: {filename}"

train_df = pd.read_csv("train_features.csv")
test_df = pd.read_csv("test_features.csv")

with open("feature_config.json") as file:
    config = json.load(file)

print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)

Training dataset: (16000, 19)
Test dataset: (4000, 19)


Validate inputs and target validate

In [ ]:
for dataset in [train_df, test_df]:
    assert dataset["device_id"].notna().all()
    assert dataset["device_id"].is_unique

assert set(train_df["device_id"]).isdisjoint(
    set(test_df["device_id"])
)

assert len(train_df) == 16000
assert len(test_df) == 4000

all_features = config["feature_columns"]
raw_features = config["raw_feature_columns"]

forbidden_inputs = {
    "device_id",
    "item_name",
    "total_profit_usd",
    "Batch_Priority"
}

assert not set(all_features) & forbidden_inputs

X = train_df[all_features].copy()
y_reg = train_df["total_profit_usd"].copy()
y_cls = train_df["Batch_Priority"].copy()

assert np.isfinite(y_reg).all()
assert y_cls.notna().all()
assert set(y_cls.unique()) == {"Low", "Medium", "High"}

policy = config["priority_policy"]

expected_labels = np.where(
    y_reg <= policy["low_cutoff_usd"],
    "Low",
    np.where(
        y_reg <= policy["high_cutoff_usd"],
        "Medium",
        "High"
    )
)

assert np.array_equal(
    y_cls.to_numpy(),
    expected_labels
)

print("Consistency checks passed.")
print("Training inputs:", X.shape)

Consistency checks passed.
Training inputs: (16000, 15)


Preprocessing pipeline

In [ ]:
def make_boosting_pipeline(model, selected_features):
    numeric = [
        column
        for column in config["numeric_features"]
        if column in selected_features
    ]

    categorical = [
        column
        for column in config["categorical_features"]
        if column in selected_features
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

create cv folders

In [ ]:
regression_cv = list(
    KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ).split(X, y_reg)
)

classification_cv = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ).split(X, y_cls)
)

feature_sets = {
    "raw_only": raw_features,
    "with_engineered": all_features
}

print("Five-fold CV configured.")

Five-fold CV configured.


Regression tune

In [ ]:
regression_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [2, 3]
}

regression_searches = {}
regression_rows = []

for name, selected_features in feature_sets.items():
    print("Running regression:", name)

    pipeline = make_boosting_pipeline(
        GradientBoostingRegressor(
            learning_rate=0.1,
            min_samples_leaf=5,
            random_state=42
        ),
        selected_features
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=regression_grid,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        refit="mae",
        cv=regression_cv,
        n_jobs=2,
        pre_dispatch=2,
        return_train_score=True,
        error_score="raise",
        verbose=1
    )

    search.fit(X, y_reg)
    regression_searches[name] = search

    results = search.cv_results_
    index = search.best_index_

    regression_rows.append({
        "feature_set": name,
        "CV_MAE_USD": -results["mean_test_mae"][index],
        "CV_MAE_STD": results["std_test_mae"][index],
        "CV_RMSE_USD": -results["mean_test_rmse"][index],
        "CV_R2": results["mean_test_r2"][index],
        "Train_MAE_USD": -results["mean_train_mae"][index],
        "best_parameters": json.dumps(search.best_params_)
    })

    pd.DataFrame(results).to_csv(
        f"07_regression_candidates_{name}.csv",
        index=False
    )

regression_comparison = pd.DataFrame(
    regression_rows
).sort_values("CV_MAE_USD")

display(regression_comparison)

best_reg_feature_set = (
    regression_comparison.iloc[0]["feature_set"]
)

best_reg_search = regression_searches[
    best_reg_feature_set
]

best_reg_model = best_reg_search.best_estimator_

print("Selected regression features:", best_reg_feature_set)
print("Selected parameters:", best_reg_search.best_params_)

Running regression: raw_only
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Running regression: with_engineered
Fitting 5 folds for each of 4 candidates, totalling 20 fits


,feature_set,CV_MAE_USD,CV_MAE_STD,CV_RMSE_USD,CV_R2,Train_MAE_USD,best_parameters
0,raw_only,0.331479,0.006211,0.613501,0.997758,0.299424,"{""model__max_depth"": 3, ""model__n_estimators"":..."
1,with_engineered,0.339311,0.005821,0.621837,0.997695,0.302788,"{""model__max_depth"": 3, ""model__n_estimators"":..."


Selected regression features: raw_only
Selected parameters: {'model__max_depth': 3, 'model__n_estimators': 200}


Classification tune

In [ ]:
classification_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [2, 3]
}

classification_searches = {}
classification_rows = []

for name, selected_features in feature_sets.items():
    print("Running classification:", name)

    pipeline = make_boosting_pipeline(
        GradientBoostingClassifier(
            learning_rate=0.1,
            min_samples_leaf=5,
            random_state=42
        ),
        selected_features
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=classification_grid,
        scoring={
            "macro_f1": "f1_macro",
            "balanced_accuracy": "balanced_accuracy",
            "accuracy": "accuracy"
        },
        refit="macro_f1",
        cv=classification_cv,
        n_jobs=2,
        pre_dispatch=2,
        return_train_score=True,
        error_score="raise",
        verbose=1
    )

    search.fit(X, y_cls)
    classification_searches[name] = search

    results = search.cv_results_
    index = search.best_index_

    classification_rows.append({
        "feature_set": name,
        "CV_Macro_F1": results["mean_test_macro_f1"][index],
        "CV_Macro_F1_STD": results["std_test_macro_f1"][index],
        "CV_Balanced_Accuracy": (
            results["mean_test_balanced_accuracy"][index]
        ),
        "CV_Accuracy": results["mean_test_accuracy"][index],
        "Train_Macro_F1": results["mean_train_macro_f1"][index],
        "best_parameters": json.dumps(search.best_params_)
    })

    pd.DataFrame(results).to_csv(
        f"07_classification_candidates_{name}.csv",
        index=False
    )

classification_comparison = pd.DataFrame(
    classification_rows
).sort_values("CV_Macro_F1", ascending=False)

display(classification_comparison)

best_cls_feature_set = (
    classification_comparison.iloc[0]["feature_set"]
)

best_cls_search = classification_searches[
    best_cls_feature_set
]

best_cls_model = best_cls_search.best_estimator_

print("Selected classification features:", best_cls_feature_set)
print("Selected parameters:", best_cls_search.best_params_)

Running classification: raw_only
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Running classification: with_engineered
Fitting 5 folds for each of 4 candidates, totalling 20 fits


,feature_set,CV_Macro_F1,CV_Macro_F1_STD,CV_Balanced_Accuracy,CV_Accuracy,Train_Macro_F1,best_parameters
0,raw_only,0.982945,0.002168,0.982936,0.982938,0.994235,"{""model__max_depth"": 3, ""model__n_estimators"":..."
1,with_engineered,0.982070,0.002304,0.982060,0.982062,0.994735,"{""model__max_depth"": 3, ""model__n_estimators"":..."


Selected classification features: raw_only
Selected parameters: {'model__max_depth': 3, 'model__n_estimators': 200}


Gradient Boosting Classification using raw features achieved a mean five-fold CV macro-F1 of 0.982945, balanced accuracy of 0.982936, and accuracy of 98.2938%. The selected configuration used 200 boosting stages and a maximum tree depth of three. Its training–CV macro-F1 gap was 0.011290. It achieved the highest mean CV macro-F1 among the four evaluated classifier families, making it the leading classification candidate. These results concern derived profit tiers; held-out test evaluation remains pending.

Fold results

In [ ]:
reg_results = best_reg_search.cv_results_
reg_index = best_reg_search.best_index_

regression_fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE_USD": [
        -reg_results[f"split{i}_test_mae"][reg_index]
        for i in range(5)
    ],
    "RMSE_USD": [
        -reg_results[f"split{i}_test_rmse"][reg_index]
        for i in range(5)
    ],
    "R2": [
        reg_results[f"split{i}_test_r2"][reg_index]
        for i in range(5)
    ]
})

cls_results = best_cls_search.cv_results_
cls_index = best_cls_search.best_index_

classification_fold_results = pd.DataFrame({
    "Fold": range(1, 6),
    "Macro_F1": [
        cls_results[f"split{i}_test_macro_f1"][cls_index]
        for i in range(5)
    ],
    "Balanced_Accuracy": [
        cls_results[f"split{i}_test_balanced_accuracy"][cls_index]
        for i in range(5)
    ],
    "Accuracy": [
        cls_results[f"split{i}_test_accuracy"][cls_index]
        for i in range(5)
    ]
})

display(regression_fold_results)
display(classification_fold_results)

,Fold,MAE_USD,RMSE_USD,R2
0,1,0.328466,0.609983,0.997863
1,2,0.338649,0.628345,0.997619
2,3,0.338419,0.632593,0.997594
3,4,0.322568,0.596777,0.997791
4,5,0.329291,0.599809,0.997922


,Fold,Macro_F1,Balanced_Accuracy,Accuracy
0,1,0.984404,0.984378,0.984375
1,2,0.983751,0.983750,0.983750
2,3,0.978746,0.978746,0.978750
3,4,0.984694,0.984685,0.984688
4,5,0.983130,0.983119,0.983125


Training CV/Gap

In [ ]:
reg_train_mae = -reg_results[
    "mean_train_mae"
][reg_index]

reg_cv_mae = -reg_results[
    "mean_test_mae"
][reg_index]

cls_train_f1 = cls_results[
    "mean_train_macro_f1"
][cls_index]

cls_cv_f1 = cls_results[
    "mean_test_macro_f1"
][cls_index]

overfitting_summary = pd.DataFrame([
    {
        "task": "Regression",
        "metric": "MAE USD",
        "training_score": reg_train_mae,
        "cv_score": reg_cv_mae,
        "gap": reg_cv_mae - reg_train_mae
    },
    {
        "task": "Classification",
        "metric": "Macro-F1",
        "training_score": cls_train_f1,
        "cv_score": cls_cv_f1,
        "gap": cls_train_f1 - cls_cv_f1
    }
])

display(overfitting_summary)

,task,metric,training_score,cv_score,gap
0,Regression,MAE USD,0.299424,0.331479,0.032054
1,Classification,Macro-F1,0.994235,0.982945,0.011290


Model summary

In [ ]:
model_summary = {
    "notebook": "07_Gradient_Boosting_Models",
    "regression_model": "GradientBoostingRegressor",
    "classification_model": "GradientBoostingClassifier",

    "regression_feature_set": best_reg_feature_set,
    "classification_feature_set": best_cls_feature_set,

    "regression_features": feature_sets[best_reg_feature_set],
    "classification_features": feature_sets[best_cls_feature_set],

    "regression_cv_mae_usd": float(
        -best_reg_search.best_score_
    ),
    "regression_cv_rmse_usd": float(
        -reg_results["mean_test_rmse"][reg_index]
    ),
    "regression_cv_r2": float(
        reg_results["mean_test_r2"][reg_index]
    ),

    "classification_cv_macro_f1": float(
        best_cls_search.best_score_
    ),
    "classification_cv_balanced_accuracy": float(
        cls_results["mean_test_balanced_accuracy"][cls_index]
    ),
    "classification_cv_accuracy": float(
        cls_results["mean_test_accuracy"][cls_index]
    ),

    "regression_parameters": {
        **best_reg_search.best_params_,
        "model__learning_rate": 0.1,
        "model__min_samples_leaf": 5
    },
    "classification_parameters": {
        **best_cls_search.best_params_,
        "model__learning_rate": 0.1,
        "model__min_samples_leaf": 5
    },

    "classification_weighting": "None",
    "random_state": 42,
    "test_evaluated": False,
    "priority_meaning": "Profit-tier proxy",
    "sklearn_version": sklearn.__version__
}

display(pd.DataFrame([model_summary]))

,notebook,regression_model,classification_model,regression_feature_set,classification_feature_set,regression_features,classification_features,regression_cv_mae_usd,regression_cv_rmse_usd,regression_cv_r2,classification_cv_macro_f1,classification_cv_balanced_accuracy,classification_cv_accuracy,regression_parameters,classification_parameters,classification_weighting,random_state,test_evaluated,priority_meaning,sklearn_version
0,07_Gradient_Boosting_Models,GradientBoostingRegressor,GradientBoostingClassifier,raw_only,raw_only,"[weight_g, device_age_years, gold_yield_g, sil...","[weight_g, device_age_years, gold_yield_g, sil...",0.331479,0.613501,0.997758,0.982945,0.982936,0.982938,"{'model__max_depth': 3, 'model__n_estimators':...","{'model__max_depth': 3, 'model__n_estimators':...",None,42,False,Profit-tier proxy,1.6.1


Models and evidence save

In [ ]:
regression_comparison.to_csv(
    "07_regression_feature_comparison.csv",
    index=False
)

classification_comparison.to_csv(
    "07_classification_feature_comparison.csv",
    index=False
)

regression_fold_results.to_csv(
    "07_regression_cv_folds.csv",
    index=False
)

classification_fold_results.to_csv(
    "07_classification_cv_folds.csv",
    index=False
)

overfitting_summary.to_csv(
    "07_overfitting_summary.csv",
    index=False
)

with open("07_model_summary.json", "w") as file:
    json.dump(model_summary, file, indent=2)

joblib.dump(
    best_reg_model,
    "07_boosting_profit_pipeline.joblib"
)

joblib.dump(
    best_cls_model,
    "07_boosting_priority_pipeline.joblib"
)

print("Gradient Boosting models and CV evidence saved.")
print("Test evaluation has not been performed.")

Gradient Boosting models and CV evidence saved.
Test evaluation has not been performed.


Interpretation

## Gradient Boosting interpretation

### Regression
- Selected feature set:
- Selected number of estimators and tree depth:
- Mean CV MAE, RMSE and R²:
- Comparison with previous regression models:
- Training/CV gap:

### Classification
- Selected feature set:
- Selected number of estimators and tree depth:
- Mean CV macro-F1 and balanced accuracy:
- Comparison with previous classifiers:
- Training/CV gap:

### Limitations
- CV results were used for tuning and feature selection.
- Only a limited hyperparameter grid was evaluated.
- Learning rate and minimum leaf size were fixed.
- Sequential boosting increases computational cost.
- Priority labels are derived profit tiers.
- Classifier CV is conditional on the training-derived policy.
- Final held-out test evaluation remains pending.

### AI-use declaration
Record actual AI assistance, student verification and contribution.

In [ ]:
files.download("07_model_summary.json")
files.download("07_regression_cv_folds.csv")
files.download("07_classification_cv_folds.csv")
files.download("07_overfitting_summary.csv")

files.download("07_boosting_profit_pipeline.joblib")
files.download("07_boosting_priority_pipeline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
train_df.to_csv("07_train_features.csv", index=False)
test_df.to_csv("07_test_features.csv", index=False)

files.download("07_train_features.csv")
files.download("07_test_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gradient Boosting Regression achieved a mean CV MAE of USD 0.331479, RMSE of USD 0.613501, and R² of 0.997758 using raw features. Random Forest achieved lower regression MAE and RMSE and was selected for recorded profit prediction. Gradient Boosting Classification achieved the highest mean CV macro-F1 of 0.982945 and was selected for profit-tier prediction. Both selections were made before held-out test evaluation. The classification improvement over Random Forest was small and is not claimed to be statistically significant.